# Exploratory detection-size and tiling draft — UNEXECUTED

The original title referred to a “3-step eval”; only two numbered steps are implemented, with no range estimator. All original code cells have null execution counts and no saved outputs. No evidence of a completed evaluation is retained here.

The code collapses airplane, drone and helicopter into an aerial-object category. Bird predictions are counted separately; these counts are **not false-positive measurements**.

The checkpoint's historical mAP50 of approximately 0.965 belongs to AOD4 four-class aggregate validation and does not validate these experiments. See `NOTEBOOK_REVIEW.md` before interpreting any output.


## Publication edits

Outputs and notebook metadata were removed; private storage paths were replaced with relative paths. The hard-coded third-party archive locator was replaced with a user-supplied environment variable. Missing dependency declarations were added, but not tested. Misleading validation and false-positive language was corrected. The exploratory algorithms are retained for inspection, with limitations disclosed; they have not been converted into a validated evaluation protocol.



In [ ]:
# Cell 0 — deps
%pip install -q ultralytics supervision datasets gdown opencv-python
import os, csv, sys, math, glob, zipfile
import numpy as np
import supervision as sv
from ultralytics import YOLO
from collections import defaultdict
print('supervision', sv.__version__)

In [ ]:
# Cell 1 — model + aerial-object collapse (mirror of cv/detector/model.py)
MODEL_PATH = 'best.pt'
AERIAL_OBJECT  = {'airplane', 'drone', 'helicopter'}   # collapse -> one alert
BIRD_CLASSES = {'bird'}  # predicted class count, not a false-positive metric

model = YOLO(MODEL_PATH)
print('MODEL CLASS NAMES:', model.names)
print('expected -> {0: airplane, 1: bird, 2: drone, 3: helicopter}')

def label_kind(name):
    if name in AERIAL_OBJECT:  return 'aerial_object'
    if name in BIRD_CLASSES: return 'bird_prediction'
    return 'other'

In [ ]:
# Cell 2 — tiling inference (mirror of cv/detector/tiling.py)
def tiled_infer(model, frame, conf, n):
    if n <= 1:
        r = model(frame, conf=conf, verbose=False)[0]
        return sv.Detections.from_ultralytics(r)
    h, w = frame.shape[:2]
    th, tw = h // n, w // n
    xs, cs, ks = [], [], []
    for row in range(n):
        for col in range(n):
            y0, x0 = row * th, col * tw
            y1 = h if row == n - 1 else y0 + th
            x1 = w if col == n - 1 else x0 + tw
            r = model(frame[y0:y1, x0:x1], conf=conf, verbose=False)[0]
            d = sv.Detections.from_ultralytics(r)
            if len(d) == 0:
                continue
            b = d.xyxy.copy(); b[:, [0, 2]] += x0; b[:, [1, 3]] += y0
            xs.append(b); cs.append(d.confidence); ks.append(d.class_id)
    if not xs:
        return sv.Detections.empty()
    m = sv.Detections(xyxy=np.vstack(xs), confidence=np.concatenate(cs),
                      class_id=np.concatenate(ks))
    return m.with_nms(threshold=0.5)

## Step 1 — exploratory image-level counts by largest annotated box

Assumes `objects.bbox` contains COCO xywh boxes; schema, units, categories and split independence have not been verified. Uses the largest annotated box per image and counts any aerial-object prediction anywhere in that image. It does not match detections to ground truth or establish a drone detection floor. Bird predictions are class counts only.


In [ ]:
# Cell 3 — Step 1: pathikg detection-floor eval
from datasets import load_dataset

CONF = 0.25
N_IMAGES = 500          # bump up once it looks sane

ds = load_dataset('pathikg/drone-detection-dataset', split='test', cache_dir='hf_cache')
print(f'test split: {len(ds)} images; evaluating first {N_IMAGES}')

buckets = [(0,10),(10,20),(20,40),(40,80),(80,160),(160,10000)]
def bucket_of(px):
    for lo,hi in buckets:
        if lo <= px < hi: return f'{lo}-{hi if hi<10000 else "inf"}px'
    return 'na'

by_size = defaultdict(lambda: {'gt':0,'hit':0})
bird_predictions = 0

for i in range(min(N_IMAGES, len(ds))):
    row = ds[i]
    img = np.array(row['image'].convert('RGB'))[:, :, ::-1]   # PIL->BGR
    gt_boxes = row['objects']['bbox']                          # COCO xywh
    gt_px = max([max(b[2], b[3]) for b in gt_boxes], default=0)
    det = tiled_infer(model, img, CONF, 1)
    kinds = [label_kind(model.names[int(c)]) for c in det.class_id] if len(det) else []
    got_aerial = 'aerial_object' in kinds
    bird_predictions += kinds.count('bird_prediction')
    if gt_px > 0:
        b = bucket_of(gt_px)
        by_size[b]['gt']  += 1
        by_size[b]['hit'] += 1 if got_aerial else 0

print('\n=== Step 1: detection rate vs. ground-truth drone size ===')
for lo,hi in buckets:
    b = f'{lo}-{hi if hi<10000 else "inf"}px'
    v = by_size[b]
    if v['gt']:
        print(f'  {b:>10}: {v["hit"]}/{v["gt"]} detected ({100*v["hit"]/v["gt"]:.0f}%)')
print(f'\nbird predictions across {min(N_IMAGES, len(ds))} imgs: {bird_predictions}')
print('Exploratory image-level counts only: no box matching, class filtering of ground truth, or false-positive estimate.')

## Step 2 — exploratory tiling sweep on one clip

The code compares frames containing any aerial prediction for 1×1, 3×3 and 6×6 grids (1, 9 and 36 inference calls per frame). It does not read the per-frame ground-truth annotations. An increase in prediction frequency does not establish better recall, precision, or validated tiling performance.


In [ ]:
# Cell 4 — download + unzip Anti-UAV300 to local disk (once)
import gdown
ANTIUAV_ID = os.environ['ANTIUAV_GOOGLE_DRIVE_ID']  # supply an authorized source yourself
os.makedirs('antiuav', exist_ok=True)
zip_path = 'antiuav/antiuav300.zip'
if not os.path.exists(zip_path):
    gdown.download(id=ANTIUAV_ID, output=zip_path)  # older gdown has no fuzzy arg; id= is enough
if not os.path.isdir('antiuav/extracted'):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('antiuav/extracted')
rgb_videos = (glob.glob('antiuav/extracted/**/*RGB*.mp4', recursive=True) or
              glob.glob('antiuav/extracted/**/visible*.mp4', recursive=True) or
              glob.glob('antiuav/extracted/**/*.mp4', recursive=True))
print(f'found {len(rgb_videos)} candidate videos')
for v in rgb_videos[:10]:
    print(' ', v)

In [ ]:
# Cell 5 — Step 2: tile 1/3/6 sweep on one HD clip (mirror of cv/eval/sweep.py)
import cv2

def sweep_video(path, tiles=(1,3,6), conf=0.25, fps=5, max_frames=200):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        print('cannot open', path); return
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30
    interval = max(1, int(src_fps / fps))
    stats = {n: {'frames':0,'aerial':0,'fp':0,'px':[]} for n in tiles}
    fn = 0; processed = 0
    while processed < max_frames:
        ret, frame = cap.read()
        if not ret: break
        fn += 1
        if fn % interval: continue
        processed += 1
        for n in tiles:
            det = tiled_infer(model, frame, conf, n)
            stats[n]['frames'] += 1
            if len(det) == 0: continue
            kinds = [label_kind(model.names[int(c)]) for c in det.class_id]
            if 'aerial_object' in kinds:
                stats[n]['aerial'] += 1
                sides = np.maximum(det.xyxy[:,2]-det.xyxy[:,0], det.xyxy[:,3]-det.xyxy[:,1])
                stats[n]['px'].append(float(sides.max()))
            stats[n]['fp'] += kinds.count('bird_prediction')
    cap.release()
    print(f'\n=== {os.path.basename(path)} ({processed} frames) ===')
    for n in tiles:
        s = stats[n]; tot = max(1, s['frames'])
        med = np.median(s['px']) if s['px'] else 0
        print(f'  tile {n}x{n}: aerial {s["aerial"]}/{tot} ({100*s["aerial"]/tot:.0f}%) '
              f'| median size {med:.0f}px | bird predictions {s["fp"]}')
    print('Exploratory prediction frequency only; ground-truth matching is required to evaluate tiling performance.')
    return stats

if rgb_videos:
    sweep_video(rgb_videos[0], tiles=(1,3,6), conf=0.25, fps=5, max_frames=200)
else:
    print('no video found — check the unzip in Cell 4')

## Review limitations and next work

- This is an unexecuted exploratory draft, not a benchmark result or field-reliability claim.
- Verify dataset schema, category mapping, bbox units, rights, authorized download source, and overlap with AOD4 before use. No evaluation datasets are redistributed.
- Step 1 is an image-level heuristic using the largest annotated box; there is no IoU matching or per-object recall calculation.
- Step 2 uses the first candidate video and at most 200 temporally sampled frames. Its glob fallback can select an unintended modality; resolution and RGB identity are not checked. Ground-truth `exist` and `gt_rect` are not consumed.
- Bird predictions cannot estimate false positives. False positives require an explicit target definition and comparison against ground truth, including negative frames.
- The reported predicted-size statistic can include bird boxes when an aerial prediction is also present. It is not ground-truth drone size or physical range.
- Tiles do not overlap. Boundary effects and additional compute confound comparisons. Dependencies are not version-pinned, and runtime compatibility has not been tested.
- The archive extraction step assumes a trusted archive. Review its contents before extraction.
- A valid follow-up would define a protocol in advance, match boxes to annotations, measure precision/recall and false positives, and account for correlated video frames. This notebook does not establish on-device behavior.
